# Pack 06 — Pulse Shaping: Raised Cosine, RRC, ISI, Nyquist Criterion, Eye Diagrams

**Module 05-dsp · Etherstack interview prep · C++ exercises in `code/`, plots and numbers here.**

| | |
|---|---|
| Time | ~2 h |
| You need | Python 3 + NumPy + Matplotlib; CMake + C++17 |
| Signals | synthetic only (random ±1 data from a seeded generator) |
| Etherstack relevance | P25 / DMR / TETRA transmitters shape their symbols with (root-)raised-cosine-type filters so the occupied spectrum fits a 12.5 kHz / 25 kHz channel mask; receivers use the matching filter. **Roll-off figures quoted below for the standards are VERIFY items.** |

## 1. Concrete first: rectangular pulses are a spectral disaster

A symbol stream sent as rectangular pulses has a sinc-shaped spectrum with sidelobes falling only 6 dB/octave — far too wide for a narrow channel. Try it: BPSK symbols at 8 samples per symbol, rectangular vs raised-cosine shaped, and compare the spectra.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
nsym = 4000
sps = 8
a = rng.choice([-1.0, 1.0], nsym)

def rc_taps(beta, sps, span):
    """Raised cosine, peak 1 at centre, span symbols each side."""
    t = np.arange(-span * sps, span * sps + 1) / sps
    h = np.zeros_like(t)
    for i, ti in enumerate(t):
        if abs(ti) < 1e-12:
            h[i] = 1.0
        elif beta > 0 and abs(abs(ti) - 1 / (2 * beta)) < 1e-9:
            h[i] = (np.pi / 4) * np.sinc(1 / (2 * beta))
        else:
            h[i] = np.sinc(ti) * np.cos(np.pi * beta * ti) / (1 - (2 * beta * ti) ** 2)
    return h

def rrc_taps(beta, sps, span):
    """Root raised cosine normalised to unit energy."""
    t = np.arange(-span * sps, span * sps + 1) / sps
    h = np.zeros_like(t)
    for i, ti in enumerate(t):
        if abs(ti) < 1e-12:
            h[i] = 1 - beta + 4 * beta / np.pi
        elif beta > 0 and abs(abs(ti) - 1 / (4 * beta)) < 1e-9:
            h[i] = (beta / np.sqrt(2)) * ((1 + 2 / np.pi) * np.sin(np.pi / (4 * beta)) + (1 - 2 / np.pi) * np.cos(np.pi / (4 * beta)))
        else:
            num = np.sin(np.pi * ti * (1 - beta)) + 4 * beta * ti * np.cos(np.pi * ti * (1 + beta))
            den = np.pi * ti * (1 - (4 * beta * ti) ** 2)
            h[i] = num / den
    return h / np.sqrt(np.sum(h ** 2))

up = np.zeros(nsym * sps); up[::sps] = a
x_rect = np.repeat(a, sps)
x_rc = np.convolve(up, rc_taps(0.35, sps, 8))

def psd_db(x, nfft=4096):
    segs = [x[i:i + nfft] * np.hanning(nfft) for i in range(0, len(x) - nfft, nfft // 2)]
    P = np.mean([np.abs(np.fft.fftshift(np.fft.fft(s))) ** 2 for s in segs], axis=0)
    return 10 * np.log10(P / P.max())

f = np.fft.fftshift(np.fft.fftfreq(4096)) * sps      # frequency in units of the symbol rate
plt.figure(figsize=(9, 4))
plt.plot(f, psd_db(x_rect), label="rectangular pulses")
plt.plot(f, psd_db(x_rc), label="raised cosine, beta = 0.35")
plt.xlim(-4, 4); plt.ylim(-100, 5); plt.xlabel("frequency [symbol rates]"); plt.ylabel("PSD [dB]")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## 2. Intersymbol interference (ISI) and the Nyquist criterion

### 2.1 Where ISI comes from
Transmit $s(t) = \sum_k a_k\, g(t - kT)$. At the receiver sample at $t = nT$:

$$s(nT) = a_n g(0) + \underbrace{\sum_{k\ne n} a_k\, g((n-k)T)}_{\text{ISI}}.$$

**Zero ISI** needs $g(mT) = 0$ for every non-zero integer $m$ (and $g(0)=1$). Many pulses do this in the time domain: a sinc does (zeros at multiples of $T$), a rectangle of width $T$ does, a half-sine does.

### 2.2 Nyquist's criterion in the frequency domain (derivation)
Sample $g(t)$ at $t = mT$: $g_s(t) = g(t)\sum_m \delta(t - mT)$. In frequency this is the spectrum aliased/summed at multiples of the symbol rate:

$$\mathcal F\{g_s\}(f) = \frac1T\sum_{k=-\infty}^{\infty} G\!\left(f - \frac kT\right).$$

Requiring $g(mT) = \delta_{m,0}$ means $g_s(t) = \delta(t)$, whose transform is the constant 1. Therefore

$$\boxed{\ \sum_{k} G\!\left(f - \tfrac kT\right) = T \quad \text{for all } f\ }.$$

**The shifted copies of $G(f)$ must add up to a constant.** The minimum-bandwidth solution is a rectangle of width $1/T$ (the sinc pulse), giving bandwidth $R_s/2$ — but its impulse response decays as $1/t$, so any timing error produces enormous ISI and it is non-realisable. Allow some **excess bandwidth** and the copies can overlap with *odd symmetry* about $\pm 1/(2T)$ (the Nyquist "vestigial symmetry"), which lets the response decay faster.

### 2.3 Raised cosine

$$H_{RC}(f) = \begin{cases} T & |f| \le \frac{1-\beta}{2T}\\[2pt] \frac T2\left[1+\cos\!\left(\frac{\pi T}{\beta}\Big(|f|-\frac{1-\beta}{2T}\Big)\right)\right] & \frac{1-\beta}{2T}<|f|\le\frac{1+\beta}{2T}\\[2pt] 0 & \text{otherwise}\end{cases}$$

$$h_{RC}(t) = \operatorname{sinc}\!\left(\frac tT\right)\frac{\cos(\pi\beta t/T)}{1-(2\beta t/T)^2}.$$

* $\beta\in[0,1]$ roll-off; occupied bandwidth $B = \frac{(1+\beta)}{2T} = \frac{(1+\beta)R_s}{2}$ (baseband one-sided) / $(1+\beta)R_s$ for a passband (two-sided) signal.
* **Why it satisfies Nyquist:** in the transition band the cosine edge is a half-cycle raised cosine: $\tfrac T2[1+\cos x]$ at frequency $f$ plus $\tfrac T2[1+\cos(\pi - x)] = \tfrac T2[1-\cos x]$ at the mirrored frequency about $\tfrac{1}{2T}$; the sum is $T$ ✓.
* **Time domain:** zeros at $t = mT$ because of the $\operatorname{sinc}$ factor; the extra factor $\cos(\pi\beta t/T)/(1-(2\beta t/T)^2)$ makes tails decay as $1/t^3$ for $\beta>0$ (vs $1/t$ for sinc). Larger $\beta$ ⇒ faster decay, more bandwidth, more timing-error tolerance.
* **Singularity:** at $t = \pm T/(2\beta)$ the denominator is 0; the limit is $\tfrac\pi4\operatorname{sinc}\!\big(\tfrac1{2\beta}\big)$ (use this in code!).

### 2.4 Root raised cosine (RRC) and the matched filter
Transmit **and** receive with filters whose product is Nyquist: $H_{RC}(f) = H_{RRC}(f)\cdot H_{RRC}(f)$ ⇒ $H_{RRC}=\sqrt{H_{RC}}$. The receive RRC is the **matched filter** (Pack 07) for the transmit RRC pulse, so it maximises SNR *and* the cascade has zero ISI. Noise is white *before* the receive filter; after it the noise spectrum is $N_0|H_{RRC}|^2 = N_0 H_{RC}$ (coloured), but the noise samples taken at multiples of $T$ are uncorrelated because $H_{RC}$ satisfies the Nyquist criterion — so symbol-spaced decisions see independent noise.

$$h_{RRC}(t)=\frac{\sin\big(\pi\tfrac tT(1-\beta)\big)+4\beta\tfrac tT\cos\big(\pi\tfrac tT(1+\beta)\big)}{\pi\tfrac tT\Big(1-\big(4\beta\tfrac tT\big)^2\Big)}\quad\text{(times }1/T\text{)}$$

Special points: $t=0$: $1-\beta+4\beta/\pi$; $t=\pm T/(4\beta)$: $\tfrac{\beta}{\sqrt2}\Big[\big(1+\tfrac2\pi\big)\sin\tfrac{\pi}{4\beta}+\big(1-\tfrac2\pi\big)\cos\tfrac\pi{4\beta}\Big]$. **RRC by itself is *not* zero-ISI**; only RRC ∗ RRC is.

The code cell below verifies the claim numerically: convolving the sampled RRC with itself gives a pulse equal to the RC pulse (up to scaling) with zeros at multiples of $T$.

In [ ]:
sps = 8; beta = 0.35; span = 16
rc = rc_taps(beta, sps, span)
rrc = rrc_taps(beta, sps, span)
pair = np.convolve(rrc, rrc)                 # length 2*len-1, centre at index len-1
c = len(rrc) - 1
print("centre value of rrc*rrc (unit energy -> 1):", round(pair[c], 6))
idx = c + sps * np.arange(-6, 7)
print("samples at multiples of T:", np.round(pair[idx], 5))
print("max |ISI| over +-1..+8 symbols:", np.max(np.abs(pair[c + sps * np.r_[1:9]])))

t = (np.arange(len(rc)) - span * sps) / sps
plt.figure(figsize=(9, 3.8))
plt.plot(t, rc / rc.max(), label="raised cosine h_RC")
plt.plot(np.arange(len(pair)) / sps - (len(rrc) - 1) / sps, pair / pair.max(), "--", label="RRC * RRC")
plt.plot(t, rrc / rrc.max(), ":", label="RRC alone")
plt.stem(np.arange(-6, 7), np.zeros(13), linefmt="k:", markerfmt="ko", basefmt=" ")
plt.xlim(-6, 6); plt.xlabel("t / T"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

### 2.5 Effect of the roll-off $\beta$ (numerical demo)
Smaller $\beta$ ⇒ narrower spectrum but longer, slower-decaying tails (higher sensitivity to timing error). Compare $\beta = 0.2, 0.35, 0.5$ (P25 / DMR / TETRA-class systems use values in this range — **VERIFY the exact roll-off of each standard**: I recall DMR and P25 transmit shaping near 0.2 and TETRA 0.35, but this is from memory).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for beta_ in (0.2, 0.35, 0.5):
    h = rc_taps(beta_, 8, 16)
    t = (np.arange(len(h)) - 16 * 8) / 8
    ax[0].plot(t, h, label=f"beta={beta_}")
    H = np.abs(np.fft.fftshift(np.fft.fft(h, 4096)))
    f = np.fft.fftshift(np.fft.fftfreq(4096)) * 8
    ax[1].plot(f, 20 * np.log10(H / H.max() + 1e-12), label=f"beta={beta_}")
ax[0].set_xlim(-5, 5); ax[0].set_title("time domain"); ax[0].legend(); ax[0].grid(True)
ax[1].set_xlim(0, 1.5); ax[1].set_ylim(-80, 3); ax[1].set_title("|H(f)| vs f/Rs"); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

## 3. Eye diagrams

### 3.1 Concrete first
Fold the received waveform into windows of **two symbol periods** and overlay them. If the system is ISI-free and timing is right, all traces pass through ±1 at the centre: the eye is **wide open**.

*Eye metrics:* **vertical opening** (height) at the best sampling time (smaller = less noise margin), **horizontal opening** (timing tolerance), zero-crossing jitter, ISI-induced thickness of the rails.

In [ ]:
def eye_traces(y, sps, offset, ntrace=300):
    """Windows of 2 symbols (2*sps+1 points) starting at offset, offset+sps, ..."""
    out, start = [], offset
    while start + 2 * sps + 1 <= len(y) and len(out) < ntrace:
        out.append(y[start:start + 2 * sps + 1])
        start += sps
    return out

def eye_plot(ax, y, delay, title, skip_symbols=60):
    # the ideal sampling instant of symbol k is at index delay + k*sps; put it in the MIDDLE of the window
    offset = (delay - sps) % sps + skip_symbols * sps
    for tr in eye_traces(y, sps, offset):
        ax.plot(np.arange(len(tr)) / sps - 1, tr, "b", alpha=0.08)
    ax.set_title(title); ax.set_xlabel("t / T  (0 = sampling instant)"); ax.grid(True)

taps = rrc_taps(0.35, sps, 8)
y_rrc = np.convolve(np.convolve(up, taps), taps)                    # RRC -> RRC (matched) = RC
y_rect = np.convolve(np.repeat(a, sps), np.ones(sps) / sps)          # rectangular Tx -> boxcar Rx
y_rcn = np.convolve(up, rc_taps(0.35, sps, 8)) + 0.1 * rng.standard_normal(len(up) + 16 * sps)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
eye_plot(ax[0], y_rrc, 16 * sps, "RRC -> RRC (matched), beta=0.35")
eye_plot(ax[1], y_rect, sps - 1, "rectangular Tx -> boxcar Rx")
eye_plot(ax[2], y_rcn, 8 * sps, "RC beta=0.35 + noise (sigma 0.1)")
plt.tight_layout(); plt.show()

### 3.2 Reading timing sensitivity numerically
For zero-ISI RC the raw ISI at a timing error $\tau$ is $\sum_{m\neq0} |h(\tau + mT)|$. At $\tau = T/2$ with $\beta = 0.35$: main term $h(0.5T)\approx 0.619$, the neighbour $h(-0.5T)\approx 0.619$ (same size!) and $h(1.5T)\approx -0.162$. A worst-case data pattern flips the neighbour's sign so the eye height $2\,[h_0 - \sum|h_m|] < 0$ — **the eye is closed at the half-symbol point**, as it should be: that is where the zero-crossings live.

In [ ]:
def rc_val(t, beta):
    t = np.asarray(t, float)
    out = np.empty_like(t)
    for i, ti in enumerate(t):
        if abs(ti) < 1e-12: out[i] = 1.0
        elif abs(abs(ti) - 1 / (2 * beta)) < 1e-9: out[i] = (np.pi / 4) * np.sinc(1 / (2 * beta))
        else: out[i] = np.sinc(ti) * np.cos(np.pi * beta * ti) / (1 - (2 * beta * ti) ** 2)
    return out

for beta_ in (0.2, 0.35, 0.5, 1.0):
    taus = np.linspace(0, 0.5, 11)
    heights = []
    for tau in taus:
        m = np.arange(-20, 21)
        h = rc_val(tau + m, beta_)
        h0 = rc_val(np.array([tau]), beta_)[0]
        heights.append(2 * (h0 - (np.sum(np.abs(h)) - abs(h0))))
    print(f"beta={beta_:4.2f}  worst-case eye height at tau/T = 0, 0.1, 0.2, 0.3, 0.4, 0.5:",
          np.round([heights[i] for i in (0, 2, 4, 6, 8, 10)], 3))

## 4. Misconceptions

| Wrong | Right |
|---|---|
| "RRC at the transmitter gives zero ISI" | Only the **cascade** Tx-RRC × Rx-RRC is Nyquist |
| "Bandwidth = symbol rate" | Occupied (two-sided passband) bandwidth = $(1+\beta)R_s$ |
| "Sinc pulses are ideal, use them" | Zero ISI only with perfect timing; $1/t$ tails → disaster with jitter, and non-causal |
| "Truncating the filter is harmless" | Short spans raise sidelobes and residual ISI; span ≥ 6–8 symbols for $\beta\ge 0.2$ |
| "Eye closed = broken receiver" | The half-symbol phase of a good eye is *supposed* to be closed; open it by sampling at the right time (Gardner, Pack 07) |
| "A matched filter for RRC must be different" | The matched filter is the time-reverse of the transmit pulse; RRC is symmetric, so it is the same filter |
| "More roll-off is always better" | Better timing tolerance, worse spectral efficiency; adjacent-channel mask limits it |

## 5. Etherstack / radio context
* **Transmit mask:** the RRC roll-off and span choose how much energy leaks into the adjacent 12.5 kHz channel. With $R_s$ = 4800 sym/s, $\beta$ = 0.2 gives a two-sided shaped-spectrum width of $(1+\beta)R_s = 5.76$ kHz (for the *pulses*; 4FSK deviation adds to it — Pack 05). **VERIFY** actual filter definitions in the standards (P25 and DMR specify the shaping explicitly).
* **Interviews:** "Why root-raised-cosine?", "What is the matched filter for this pulse?", "How do you test a modem? — eye diagram + constellation + BER".
* **Implementation:** an RRC has to be *truncated* (span) and quantised; both determine residual ISI. At 4–8 samples/symbol and span 8, that is 65–129 taps — a polyphase interpolating filter (Pack 04) makes it cheap.

## 6. Exercises

### Part A — pen and paper
1. A system sends 4800 sym/s with $\beta=0.2$ RRC shaping. Occupied (passband, two-sided) bandwidth?
2. Show that the sinc pulse $g(t)=\mathrm{sinc}(t/T)$ satisfies the Nyquist criterion in time, and sketch $G(f)$.
3. Verify the RC time-domain formula at $t = T$ for $\beta=0.35$ gives 0, and compute $h_{RC}(0.5T)$ (use the formula). Then compute the limit value at $t = T/(2\beta)$ for $\beta = 0.5$.
4. For $\beta=0.35$ compute worst-case peak ISI (sum of |h|) at $\tau=0.1T$ using terms $m=-2..2$. (Values: $h(0.1)$, $h(-0.9)$, $h(1.1)$, $h(-1.9)$, $h(2.1)$.)
5. Why can the Nyquist criterion in frequency be satisfied by a rectangle of width $1/T$ and by the raised-cosine shape? Show that for $\beta=1$ the shapes $\tfrac T2(1+\cos(\pi f T))$ add to $T$ when shifted by $1/T$.
6. A receiver with RRC matched filter sees an eye closing from timing jitter of 15 % of T at $\beta=0.2$. Estimate roughly whether the opening is positive. (Use your numbers from Question 4 as a guide.)
7. What is the RRC at $t=0$ for $\beta=0.35$? (Formula above.) Normalised to unit energy this changes — why is unit energy preferable in code?
8. QPSK at 4800 sym/s, $\beta=0.35$: bandwidth? Compare with 4FSK ($\pm$1800 Hz outer deviation).

### Part B — C++ (`code/`)
| # | Function | Concept |
|---|---|---|
| B1 | `raised_cosine` | RC taps with singular-point handling |
| B2 | `root_raised_cosine` | RRC taps with the two special points, unit energy |
| B3 | `pulse_shape` | zero-stuff upsample + FIR |
| B4 | `eye_traces` | fold into 2-symbol windows |
| B5 | `eye_height` | vertical opening at a sampling phase |
| B6 | `isi_residual` | max off-centre |sample at multiples of T| of a pulse |

### Part C — interview questions
1. State the Nyquist ISI criterion in time and frequency and derive the frequency form.
2. Why split the raised cosine into two root-raised-cosine filters?
3. What does the roll-off factor trade off?
4. What does an eye diagram tell you that a constellation does not?
5. How would you detect from the eye that the symbol clock is slightly fast?

## 7. VERIFY list
* Roll-off factors of P25 / DMR / TETRA transmit shaping (mentioned as "around 0.2 / 0.35") — **check each standard**.
* The tail decay claims ($1/t$ for sinc, $1/t^3$ for RC with $\beta>0$) are textbook; verify numerically if required.
* Code cells in this notebook were **not executed** by the author (no Python in the authoring environment). Run them; expected checks: centre of RRC∗RRC is 1.000000; samples at ±mT are ≲ 1e-3.